In [1]:
from langchain_ollama import ChatOllama 
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv

load_dotenv()

True

### 1.LLM calling


In [2]:
llm = ChatOllama(model="llama3.2:3b" , temperature=0.7)

question = "who is the cm of tamil nadu in 2026"

llm.invoke(question)

AIMessage(content="I don't have information about the Chief Minister of Tamil Nadu for the year 2026, as my knowledge cutoff is December 2023 and I do not have the ability to predict the future or provide information that has not yet occurred. If you're looking for information on current or recent Chief Ministers of Tamil Nadu, I'd be happy to help with that!", additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-17T11:02:33.031578Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9666491000, 'load_duration': 176398500, 'prompt_eval_count': 37, 'prompt_eval_duration': 539212800, 'eval_count': 74, 'eval_duration': 8841041600, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019d9b1b-2f7d-7982-8c8b-e3b99c340dc9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 74, 'total_tokens': 111})

### 2.Tool Calling
1. Tavily for WebSearch , 
2. Customize summarize prompt template ,
3. Notes - topic and content .

In [3]:
@tool
def web_search(query: str) -> str :
    """Search the web and return the real time information on any topic."""
    result = TavilySearchResults(max_results=2)
    result = result.invoke({"query": query})

    output = ""
    for i in result:
        output += f"Title: {i['title']}\n"
        output += f"Content: {i['content']}\n\n"
    return output.strip()

In [4]:
r1 = web_search.invoke("latest news on tamil nadu cm in 2026")
print(r1[:500])

C:\Users\VICTUS\AppData\Local\Temp\ipykernel_11528\3361209845.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  result = TavilySearchResults(max_results=2)


Title: Tamil Nadu assembly polls 2026: Chief minister Stalin praises Jayalalithaa for opposing BJP policies; calls EPS a ‘dummy’ | Chennai News - The Times of India
Content: TOI logo

## TOI

# Tamil Nadu assembly polls 2026: Chief minister Stalin praises Jayalalithaa for opposing BJP policies; calls EPS a ‘dummy’

Tamil Nadu assembly polls 2026: Chief minister Stalin praises Jayalalithaa for opposing BJP policies; calls EPS a ‘dummy’

## Videos [...] Article image for: Poila Boishakh 2026: 9 tr


In [5]:
@tool
def summarize(text: str) -> str:
    """Summarize the given text (Long paragraph into Short Paragraph)."""
    prompt = f"Summarize the following text:\n\n{text}"
    response = llm.invoke(prompt)
    return response.content.strip()

In [6]:
result = summarize.invoke(r1)
print(result)

The article does not mention the 2026 Tamil Nadu assembly polls. Instead, it discusses an article by The Times of India that praises Chief Minister M.K. Stalin for opposing BJP policies and calls Opposition leader Edappadi K. Palaniswami (EPS) a "dummy". There is no relevant information about the 2026 elections in the text provided.


In [ ]:
@tool
def notes(text: str) -> str:
    """ use this tool to convert AI-generated responses into clean, structured notes. """
    prompt = f"""Convert the following AI-generated response into clean, structured notes.

The input is the latest response from an AI model. Do NOT add new information. Only reorganize and refine it.

Return the output in EXACT format:

Title: <short, clear topic name>
Content:
- Point 1
- Point 2
- Point 3
- (3–6 bullet points total)

Rules:
- Title must summarize the main idea (max 8–10 words)
- Content must be concise and structured as bullet points
- Do NOT include anything outside this format
- Do NOT repeat the input text verbatim — refine it
- Do NOT hallucinate or add new facts

Text:
{text}"""
    response = llm.invoke(prompt)
    return response.content.strip()

In [17]:
result = notes.invoke(r1)
print(result)

Title: MK Stalin Launches Statewide Protest Against Delimitation

Content:
- MK Stalin launches statewide anti-delimitation agitation
- Burns a copy of proposed bill against delimitation
- Protest marks opposition to bill's implementation in Tamil Nadu
- Movement seeks to highlight concerns over delimitation process


### 3.Manual Agent Loop

In [9]:
SYSTEM_PROMPT = """You are an AI agent that can use tools to answer user queries.

You MUST decide the next action and respond ONLY in valid JSON format.

Available tools:

1. web_search(query: str)
   - Use this when the question requires real-time or external information.
   - Example: latest news, current events, recent updates.

2. summarize(text: str)
   - Use this when you are given long text and need to shorten it.
   - Output should be a concise paragraph.

3. notes(text: str)
   - Use this to convert information into structured notes.
   - Output should include:
     Title: <short title>
     Content: <detailed explanation and bullet points>

---

Rules:

- Always respond in JSON format ONLY.
- Do NOT include explanations outside JSON.
- Use the format:

{
  "tool": "tool_name",
  "args": {
    "parameter_name": "value"
  }
}

- When you have enough information to answer, return:

{
  "tool": "final_answer",
  "args": {
    "answer": "your final answer here"
  }
}

---

Multi-step reasoning:

- If the query requires multiple steps:
  Example:
  "Find latest news and summarize"

  Step 1 → call web_search  
  Step 2 → call summarize  
  Step 3 → call notes 

- Always use tools when needed.
- Do NOT guess if tool can provide better info.

---

Important:

- Never return plain text.
- Never return partial JSON.
- Always return a valid JSON object."""

In [10]:
import json
tools_mapping = { "web_search": web_search, "summarize": summarize, "notes": notes}

def run_agent(user_input , max_steps=3):
     print(f"\n{'='*60}")
     print(f"User Query: {user_input}")
     print(f"{'='*60}")

     messages = [("system", SYSTEM_PROMPT), ("user", user_input)]

     for step in range(max_steps):
          
          response = llm.invoke(messages)
          raw = response.content.strip()
          print(f"\nAgent Response (Step {step+1}): {raw}\n")

          try :
            if raw.startswith("```"):
                  raw = raw.split("```")[1]
                  if raw.startswith("json"):
                        raw = raw[4:]
                  raw = raw.strip()
            decision = json.loads(raw)
          
          except json.JSONDecodeError:
                print(f"\n[Step {step+1}] LLM returned non-JSON, treating as final answer.")
                print(f"Answer: {raw}")
                return raw 
          
          tool_name = decision.get("tool", "")
          args = decision.get("args", {})


          if tool_name == "final_answer":
                answer = args.get("answer", raw)
                print(f"\n[Step {step+1}] Final Answer: {answer}")
                return answer
          
          if tool_name in tools_mapping:
                tool_func = tools_mapping[tool_name]
                tool_result = tool_func.invoke(list(args.values())[0])
                print(f"\n[Step {step+1}] Tool '{tool_name}' returned:\n{tool_result}\n")
                messages.append(("assistant", raw))
                messages.append(("user", tool_result))

          else :
                print(f"\n[Step {step+1}] Unknown tool '{tool_name}' requested. Stopping.")
                return None
          
     print("\n Max steps reached ")
     return None
           


### Test-1

In [11]:
run_agent("What is the latest news on OpenAI?")


User Query: What is the latest news on OpenAI?

Agent Response (Step 1): {
  "tool": "web_search",
  "args": {
    "query": "latest news OpenAI"
  }
}


[Step 1] Tool 'web_search' returned:
Title: OpenAI News
Content: Research Apr 16, 2026. accelerating-cyber-defense-ecosystem-1x1 ... Latest Advancements. GPT-5.3 Instant · GPT-5.3-Codex · GPT-5 · Codex. Safety. Safety

Title: OpenAI Research | Release
Content: OpenAI introduces GPT-Rosalind, a frontier reasoning model built to accelerate drug discovery, genomics analysis, protein reasoning, and scientific research


Agent Response (Step 2): {
  "tool": "notes",
  "args": {
    "text": "OpenAI News\nResearch Apr 16, 2026. accelerating-cyber-defense-ecosystem-1x1 ... Latest Advancements. GPT-5.3 Instant \\nGPT-5.3-Codex \\nGPT-5 \\nCodex.\n\nTitle: OpenAI Research | Release\\nContent: OpenAI introduces GPT-Rosalind, a frontier reasoning model built to accelerate drug discovery, genomics analysis, protein reasoning, and scientific resear

'OpenAI has introduced GPT-Rosalind, a new frontier reasoning model designed to accelerate various scientific research areas such as drug discovery, genomics analysis, and protein reasoning.'

### Test -2

In [12]:
run_agent("Find the latest news on OpenAI and summarize it in 3-4 sentences.")


User Query: Find the latest news on OpenAI and summarize it in 3-4 sentences.

Agent Response (Step 1): {
  "tool": "web_search",
  "args": {
    "query": "latest news OpenAI"
  }
}
{
  "tool": "summarize",
  "args": {
    "text": "OpenAI latest news"
  }
}


[Step 1] LLM returned non-JSON, treating as final answer.
Answer: {
  "tool": "web_search",
  "args": {
    "query": "latest news OpenAI"
  }
}
{
  "tool": "summarize",
  "args": {
    "text": "OpenAI latest news"
  }
}


'{\n  "tool": "web_search",\n  "args": {\n    "query": "latest news OpenAI"\n  }\n}\n{\n  "tool": "summarize",\n  "args": {\n    "text": "OpenAI latest news"\n  }\n}'

### Test-3


In [13]:
run_agent("what is llm")


User Query: what is llm

Agent Response (Step 1): {
  "tool": "web_search",
  "args": {
    "query": "LLM definition"
  }
}


[Step 1] Tool 'web_search' returned:
Title: What is an LLM (large language model)? - Cloudflare
Content: ## What is a large language model (LLM)?

A large language model (LLM) is a type of artificial intelligence (AI) program that can recognize and generate text, among other tasks. LLMs are trained on huge sets of data — hence the name "large." LLMs are built on machine learning: specifically, a type of neural network called a transformer model. [...] In simpler terms, an LLM is a computer program that has been fed enough examples to be able to recognize and interpret human language or other types of complex data. Many LLMs are trained on data that has been gathered from the Internet — thousands or millions of gigabytes' worth of text. Some LLMs continue to crawl the web for more content after they are initially trained. But the quality of the samples impacts h

'A large language model (LLM) is an artificial intelligence program that recognizes and generates text, using machine learning and transformer models to comprehend and generate human language.'

### 4.ReAct Agent (Resoning + action)

In [ ]:
agent = create_agent(llm, tools=[web_search, summarize, notes], system_prompt=SYSTEM_PROMPT)

def run_react_agent(query):
    """Run a query through the ReAct agent and print the result."""
    print(f"\nQuery: {query}")
    print("-" * 40)

    result = agent.invoke({"messages": [("human", query)]})
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"[Tool Call] {tc['name']}({tc['args']})")
        elif msg.type == "tool":
            print(f"[Tool Result] {msg.content}...")
        elif msg.type == "ai" and msg.content:
            print(f"[Final Answer] {msg.content}")


In [19]:
run_react_agent("What is the latest news on OpenAI?")


Query: What is the latest news on OpenAI?
----------------------------------------
[Tool Call] web_search({'query': 'latest news on OpenAI'})
[Tool Result] Title: OpenAI News
Content: Research Apr 16, 2026. accelerating-cyber-defense-ecosystem-1x1 ... Latest Advancements. GPT-5.3 Instant · GPT-5.3-Codex · GPT-5 · Codex. Safety. Safety

Title: OpenAI Research | Release
Content: OpenAI introduces GPT-Rosalind, a frontier reasoning model built to accelerate drug discovery, genomics analysis, protein reasoning, and scientific research...
[Tool Call] notes({'tool': 'notes', 'args': {'text': 'OpenAI has introduced GPT-Rosalind, a new frontier reasoning model that can be used for accelerating drug discovery, genomics analysis, protein reasoning, and scientific research.'}})
[Tool Result] Error invoking tool 'notes' with kwargs {'tool': 'notes', 'args': {'text': 'OpenAI has introduced GPT-Rosalind, a new frontier reasoning model that can be used for accelerating drug discovery, genomics analy